In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv('../data/processed/train_cleaned.csv', parse_dates=['date'])
train.head()

,date,store_nbr,family,sales,onpromotion,dcoilwtico,city,state,type,cluster,transactions,is_holiday
0,2013-01-01,1,AUTOMOTIVE,0.0,0,93.14,Quito,Pichincha,D,13,0.0,1
1,2013-01-01,1,BABY CARE,0.0,0,93.14,Quito,Pichincha,D,13,0.0,1
2,2013-01-01,1,BEAUTY,0.0,0,93.14,Quito,Pichincha,D,13,0.0,1
3,2013-01-01,1,BEVERAGES,0.0,0,93.14,Quito,Pichincha,D,13,0.0,1
4,2013-01-01,1,BOOKS,0.0,0,93.14,Quito,Pichincha,D,13,0.0,1


## Time-Based Features


In [3]:
train['year'] = train['date'].dt.year
train['month'] = train['date'].dt.month
train['day'] = train['date'].dt.day
train['day_of_week'] = train['date'].dt.dayofweek
train['week_of_year'] = train['date'].dt.isocalendar().week.astype(int)
train['is_weekend'] = (train['day_of_week'] >= 5).astype(int)
train['quarter'] = train['date'].dt.quarter

print(train[['date', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'is_weekend', 'quarter']].head())

        date  year  month  day  day_of_week  week_of_year  is_weekend  quarter
0 2013-01-01  2013      1    1            1             1           0        1
1 2013-01-01  2013      1    1            1             1           0        1
2 2013-01-01  2013      1    1            1             1           0        1
3 2013-01-01  2013      1    1            1             1           0        1
4 2013-01-01  2013      1    1            1             1           0        1


## Lag Features

In [4]:
train = train.sort_values(['store_nbr', 'family', 'date'])

lag_days = [7, 14, 28]

for lag in lag_days:
    train[f'sales_lag_{lag}'] = train.groupby(['store_nbr', 'family'])['sales'].shift(lag)

print(train[['date', 'store_nbr', 'family', 'sales', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28']].head(30))

            date  store_nbr      family  sales  sales_lag_7  sales_lag_14  \
0     2013-01-01          1  AUTOMOTIVE    0.0          NaN           NaN   
1782  2013-01-02          1  AUTOMOTIVE    2.0          NaN           NaN   
3564  2013-01-03          1  AUTOMOTIVE    3.0          NaN           NaN   
5346  2013-01-04          1  AUTOMOTIVE    3.0          NaN           NaN   
7128  2013-01-05          1  AUTOMOTIVE    5.0          NaN           NaN   
8910  2013-01-06          1  AUTOMOTIVE    2.0          NaN           NaN   
10692 2013-01-07          1  AUTOMOTIVE    0.0          NaN           NaN   
12474 2013-01-08          1  AUTOMOTIVE    2.0          0.0           NaN   
14256 2013-01-09          1  AUTOMOTIVE    2.0          2.0           NaN   
16038 2013-01-10          1  AUTOMOTIVE    2.0          3.0           NaN   
17820 2013-01-11          1  AUTOMOTIVE    3.0          3.0           NaN   
19602 2013-01-12          1  AUTOMOTIVE    2.0          5.0           NaN   

## Rolling Mean Features

In [5]:
windows = [7, 14, 28]

for window in windows:
    train[f'rolling_mean_{window}'] = train.groupby(['store_nbr', 'family'])['sales'].transform(
        lambda x: x.shift(1).rolling(window).mean()
    )

print(train[['date', 'store_nbr', 'family', 'sales', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28']].head(35))

            date  store_nbr      family  sales  rolling_mean_7  \
0     2013-01-01          1  AUTOMOTIVE    0.0             NaN   
1782  2013-01-02          1  AUTOMOTIVE    2.0             NaN   
3564  2013-01-03          1  AUTOMOTIVE    3.0             NaN   
5346  2013-01-04          1  AUTOMOTIVE    3.0             NaN   
7128  2013-01-05          1  AUTOMOTIVE    5.0             NaN   
8910  2013-01-06          1  AUTOMOTIVE    2.0             NaN   
10692 2013-01-07          1  AUTOMOTIVE    0.0             NaN   
12474 2013-01-08          1  AUTOMOTIVE    2.0        2.142857   
14256 2013-01-09          1  AUTOMOTIVE    2.0        2.428571   
16038 2013-01-10          1  AUTOMOTIVE    2.0        2.428571   
17820 2013-01-11          1  AUTOMOTIVE    3.0        2.285714   
19602 2013-01-12          1  AUTOMOTIVE    2.0        2.285714   
21384 2013-01-13          1  AUTOMOTIVE    2.0        1.857143   
23166 2013-01-14          1  AUTOMOTIVE    2.0        1.857143   
24948 2013

## Encoding and Cleaning

In [6]:
# Drop rows with NaN from lag/rolling features
train = train.dropna()

train['family'] = train['family'].astype('category').cat.codes
train['city'] = train['city'].astype('category').cat.codes
train['state'] = train['state'].astype('category').cat.codes
train['type'] = train['type'].astype('category').cat.codes

# Drop date column - already extracted features from it
train = train.drop(columns=['date'])

print(train.shape)
print(train.dtypes)

(2950992, 24)
store_nbr            int64
family                int8
sales              float64
onpromotion          int64
dcoilwtico         float64
city                  int8
state                 int8
type                  int8
cluster              int64
transactions       float64
is_holiday           int64
year                 int32
month                int32
day                  int32
day_of_week          int32
week_of_year         int64
is_weekend           int64
quarter              int32
sales_lag_7        float64
sales_lag_14       float64
sales_lag_28       float64
rolling_mean_7     float64
rolling_mean_14    float64
rolling_mean_28    float64
dtype: object


In [7]:
train.to_csv('../data/processed/train_featured.csv', index=False)
print(f'Dataset saved with shape: {train.shape}')

Dataset saved with shape: (2950992, 24)
